# 09. Retrieval Metrics

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/evals/09_retrieval_metrics.ipynb)

Module 3 begins by evaluating the retriever itself. In this notebook you will build a tiny local benchmark, run dense retrieval over it, and compute ranking metrics from first principles.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **Learning goals**
- Understand **Why start here?**
- Understand **Step 1 — Define a small local corpus**
- Understand **Step 2 — Create benchmark queries and judgments**
- Understand **Step 3 — Run dense retrieval over the benchmark**


## Learning goals

- build a local retrieval benchmark with graded relevance labels
- compute recall@k, precision@k, MRR, and nDCG without eval libraries
- compare a dense retriever against a simple lexical baseline
- inspect query-level failures instead of hiding behind one average

## Why start here?

RAG quality is capped by retrieval quality. If the right evidence never enters the context window, the generator cannot faithfully answer the question. That is why good RAG evaluation starts one layer below generation.

In [ ]:
# --- Google Colab Setup ---
!pip install -q "transformers>=4.51.0" accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy

import csv
import json
import math
import random
import statistics
import time
from pathlib import Path
from typing import Any, Dict, List

import faiss
import numpy as np
import torch
from google.colab import drive
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"
EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME, cache_folder=CACHE_DIR)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True, top_k=20):
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=top_k,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

def embed_texts(texts, batch_size=32):
    if isinstance(texts, str):
        texts = [texts]
    return embed_model.encode(
        list(texts),
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

def build_faiss_index(texts):
    texts = list(texts)
    embeddings = embed_texts(texts).astype(np.float32)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return {
        "texts": texts,
        "embeddings": embeddings,
        "index": index,
    }

def search_index(query, store, k=5):
    k = min(k, len(store["texts"]))
    if k <= 0:
        return []

    query_vector = embed_texts([query]).astype(np.float32)
    scores, indices = store["index"].search(query_vector, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        results.append(
            {
                "index": int(idx),
                "score": float(score),
                "text": store["texts"][idx],
            }
        )
    return results

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"✓ Embedding model loaded: {EMBEDDING_MODEL_NAME}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## Step 1 — Define a small local corpus

The corpus below is intentionally tiny and transparent. You should be able to read every document, inspect every failure, and understand exactly why a ranking metric moves.

In [ ]:
from collections import Counter
import re

random.seed(7)

STOPWORDS = {
    "the", "a", "an", "and", "or", "to", "of", "in", "for", "on", "with", "by",
    "at", "from", "during", "into", "over", "under", "after", "before", "than",
    "what", "which", "where", "when", "how", "did", "does", "is", "are", "was",
    "were", "be", "been", "it", "its", "their", "that", "this", "these", "those"
}


def normalize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def content_words(text):
    return [token for token in normalize(text).split() if token not in STOPWORDS]


def clip(text, width=72):
    text = str(text)
    return text if len(text) <= width else text[: width - 3] + "..."


def print_rows(rows, columns=None, limit=None):
    rows = list(rows)
    if limit is not None:
        rows = rows[:limit]
    if not rows:
        print("(no rows)")
        return
    if columns is None:
        columns = list(rows[0].keys())
    widths = {}
    for column in columns:
        widths[column] = len(str(column))
        for row in rows:
            widths[column] = max(widths[column], len(str(row.get(column, ""))))
    header = " | ".join(str(column).ljust(widths[column]) for column in columns)
    divider = "-+-".join("-" * widths[column] for column in columns)
    print(header)
    print(divider)
    for row in rows:
        print(" | ".join(str(row.get(column, "")).ljust(widths[column]) for column in columns))

In [ ]:
CORPUS = [
    {
        "doc_id": "D1",
        "title": "Harbor District Microgrid Pilot",
        "text": "The Harbor district launched a resilience pilot in 2024. It installed 6 megawatts of rooftop solar and a 20 megawatt-hour battery. During two grid outages, the microgrid kept the health clinic and water pumps online for 11 hours. Operators reported a 34 percent reduction in diesel generator use compared with the previous summer.",
    },
    {
        "doc_id": "D2",
        "title": "Cedar Wastewater Heat Recovery Memo",
        "text": "The Cedar treatment plant now captures heat from outgoing effluent. The recovered heat keeps digesters warm and supplies a nearby greenhouse loop. Plant managers reported an 18 percent reduction in natural gas consumption. The memo estimates a payback period of about 5.5 years.",
    },
    {
        "doc_id": "D3",
        "title": "Northwind Offshore Wind Siting Brief",
        "text": "The Northwind team shifted turbine placement 9 kilometers east to avoid a major bird migration corridor. The export cable shares an existing shipping channel for the first 14 kilometers. Environmental review says construction noise is the largest short-term risk. The project brief estimates annual output could power 210000 homes.",
    },
    {
        "doc_id": "D4",
        "title": "Library Retrofit Case Study",
        "text": "A central library replaced chillers with ground-source heat pumps and smart ventilation controls. Energy use intensity fell from 212 to 148 kilowatt-hours per square meter. Summer comfort complaints dropped by 40 percent. Total project cost was 4.2 million dollars with an 11-year payback.",
    },
    {
        "doc_id": "D5",
        "title": "Drought Response Handbook",
        "text": "Stage two restrictions begin when reservoir storage stays below 62 percent for 21 consecutive days. The handbook says leak repairs can save up to 12 percent of summer demand. Recharge basins should rest every seventh day to prevent compaction. Public messaging focuses on irrigation schedules and the repair hotline.",
    },
    {
        "doc_id": "D6",
        "title": "Transit Electrification Plan",
        "text": "Phase one purchases 18 battery buses by 2026. The full program targets 42 electric buses by 2028. Overnight depot charging serves most vehicles, while an on-route pantograph at Central Station handles the peak corridor. The plan forecasts 22 percent lower maintenance cost than diesel operations.",
    },
    {
        "doc_id": "D7",
        "title": "Estuary Flood Barrier Memo",
        "text": "The estuary barrier closes only during storm-surge warnings above 1.7 meters. Wetlands restoration is expected to delay peak flooding by 35 minutes. The annual maintenance budget is 8.4 million dollars. Fishing access is preserved through a side channel.",
    },
    {
        "doc_id": "D8",
        "title": "Data Center Water Reuse Project",
        "text": "A reclaim system now reuses 68 percent of cooling water at the municipal data center. The facility pairs with treated greywater from the wastewater plant. Summer potable water demand fell by 41 percent after deployment. Filtration membranes are replaced every 18 months.",
    },
]

DOC_TEXTS = [doc["title"] + ". " + doc["text"] for doc in CORPUS]

print_rows(
    [
        {
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "summary": clip(doc["text"], 88),
        }
        for doc in CORPUS
    ],
    columns=["doc_id", "title", "summary"],
)

## Step 2 — Create benchmark queries and judgments

Each query gets graded relevance labels. A score of 3 means the document is the best answer source, while 1 means partially helpful but not ideal. This graded setup is what makes nDCG useful.

In [ ]:
BENCHMARK = [
    {
        "qid": "Q1",
        "query": "Which project kept critical services running during outages and reduced diesel generator use?",
        "judgments": {"D1": 3, "D7": 1},
    },
    {
        "qid": "Q2",
        "query": "What document discusses battery bus rollout and depot charging?",
        "judgments": {"D6": 3, "D1": 1},
    },
    {
        "qid": "Q3",
        "query": "Where can I find the threshold for stage two drought restrictions and recharge basin guidance?",
        "judgments": {"D5": 3, "D8": 1},
    },
    {
        "qid": "Q4",
        "query": "Which brief explains cable routing changes made to reduce ecological impact offshore?",
        "judgments": {"D3": 3, "D7": 1},
    },
    {
        "qid": "Q5",
        "query": "What case study reports energy use intensity falling from 212 to 148?",
        "judgments": {"D4": 3, "D2": 1},
    },
    {
        "qid": "Q6",
        "query": "Which project cut potable water demand through reuse and how much did it cut?",
        "judgments": {"D8": 3, "D2": 1, "D5": 1},
    },
]

print_rows(
    [
        {
            "qid": item["qid"],
            "query": clip(item["query"], 82),
            "graded_docs": ", ".join(
                "{}:{}".format(doc_id, grade)
                for doc_id, grade in item["judgments"].items()
            ),
        }
        for item in BENCHMARK
    ],
    columns=["qid", "query", "graded_docs"],
)

## Step 3 — Run dense retrieval over the benchmark

The shared setup cell already loaded an embedding model and FAISS helper functions. We will index the document texts, then inspect the ranked list for one query before scoring the whole benchmark.

In [ ]:
DOC_STORE = build_faiss_index(DOC_TEXTS)


def dense_search(query, k=5):
    results = search_index(query, DOC_STORE, k=k)
    ranked = []
    for result in results:
        doc = CORPUS[result["index"]]
        ranked.append(
            {
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "score": round(result["score"], 4),
            }
        )
    return ranked


demo_rows = dense_search(BENCHMARK[0]["query"], k=5)
print("Query:", BENCHMARK[0]["query"])
print_rows(demo_rows, columns=["doc_id", "title", "score"])
print("Expected top source:", BENCHMARK[0]["judgments"])

## Step 4 — Implement the ranking metrics from scratch

We will compute four common retrieval metrics:

- recall@k: how much of the relevant set appears in the top k?
- precision@k: how much of the top k is relevant?
- MRR: how early does the first relevant hit appear?
- nDCG: how good is the full ordering when relevance is graded?

In [ ]:
def relevant_ids(judgments):
    return [doc_id for doc_id, grade in judgments.items() if grade > 0]


def recall_at_k(ranked_ids, judgments, k):
    relevant = set(relevant_ids(judgments))
    if not relevant:
        return 0.0
    return len(set(ranked_ids[:k]) & relevant) / len(relevant)


def precision_at_k(ranked_ids, judgments, k):
    window = ranked_ids[:k]
    if not window:
        return 0.0
    relevant = set(relevant_ids(judgments))
    return len(set(window) & relevant) / len(window)


def reciprocal_rank(ranked_ids, judgments):
    relevant = set(relevant_ids(judgments))
    for rank, doc_id in enumerate(ranked_ids, start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0


def dcg_at_k(ranked_ids, judgments, k):
    total = 0.0
    for rank, doc_id in enumerate(ranked_ids[:k], start=1):
        rel = judgments.get(doc_id, 0)
        if rel > 0:
            total += (2 ** rel - 1) / math.log2(rank + 1)
    return total


def ndcg_at_k(ranked_ids, judgments, k):
    ideal_rels = sorted(judgments.values(), reverse=True)[:k]
    ideal = 0.0
    for rank, rel in enumerate(ideal_rels, start=1):
        if rel > 0:
            ideal += (2 ** rel - 1) / math.log2(rank + 1)
    if ideal == 0:
        return 0.0
    return dcg_at_k(ranked_ids, judgments, k) / ideal


toy_ranked = ["D7", "D1", "D3", "D5"]
toy_judgments = {"D1": 3, "D7": 1}

print("Toy ranking:", toy_ranked)
print("Toy judgments:", toy_judgments)
print("recall@1 =", round(recall_at_k(toy_ranked, toy_judgments, 1), 3))
print("precision@2 =", round(precision_at_k(toy_ranked, toy_judgments, 2), 3))
print("MRR =", round(reciprocal_rank(toy_ranked, toy_judgments), 3))
print("nDCG@3 =", round(ndcg_at_k(toy_ranked, toy_judgments, 3), 3))

In [ ]:
def evaluate_retriever(benchmark, search_fn, k=5):
    rows = []
    for item in benchmark:
        ranked = search_fn(item["query"], k=k)
        ranked_ids = [row["doc_id"] for row in ranked]
        rows.append(
            {
                "qid": item["qid"],
                "recall@{}".format(k): round(recall_at_k(ranked_ids, item["judgments"], k), 3),
                "precision@{}".format(k): round(precision_at_k(ranked_ids, item["judgments"], k), 3),
                "mrr": round(reciprocal_rank(ranked_ids, item["judgments"]), 3),
                "ndcg@{}".format(k): round(ndcg_at_k(ranked_ids, item["judgments"], k), 3),
                "top3": ", ".join(ranked_ids[:3]),
            }
        )
    return rows


def aggregate_metrics(rows, k=5):
    return {
        "recall@{}".format(k): round(statistics.mean(row["recall@{}".format(k)] for row in rows), 3),
        "precision@{}".format(k): round(statistics.mean(row["precision@{}".format(k)] for row in rows), 3),
        "mrr": round(statistics.mean(row["mrr"] for row in rows), 3),
        "ndcg@{}".format(k): round(statistics.mean(row["ndcg@{}".format(k)] for row in rows), 3),
    }


dense_rows = evaluate_retriever(BENCHMARK, dense_search, k=5)
print_rows(dense_rows, columns=["qid", "recall@5", "precision@5", "mrr", "ndcg@5", "top3"])
print()
print("Dense retriever averages:", aggregate_metrics(dense_rows, k=5))

## Step 5 — Add a lexical baseline

Benchmarks become useful when you compare alternatives. The baseline below is intentionally simple: count keyword matches in the title and body, then rank by overlap.

In [ ]:
LEXICAL_COUNTS = {
    doc["doc_id"]: Counter(content_words(doc["title"] + " " + doc["text"]))
    for doc in CORPUS
}


def lexical_search(query, k=5):
    query_terms = content_words(query)
    rows = []
    for doc in CORPUS:
        counts = LEXICAL_COUNTS[doc["doc_id"]]
        body_hits = sum(counts[term] for term in query_terms)
        title_hits = sum(1 for term in query_terms if term in normalize(doc["title"]).split())
        score = body_hits + 1.5 * title_hits
        rows.append(
            {
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "score": round(score, 3),
            }
        )
    rows.sort(key=lambda row: (row["score"], row["doc_id"]), reverse=True)
    return rows[:k]


print_rows(lexical_search(BENCHMARK[3]["query"], k=5), columns=["doc_id", "title", "score"])
print("Dense ranking for the same query:")
print_rows(dense_search(BENCHMARK[3]["query"], k=5), columns=["doc_id", "title", "score"])

In [ ]:
systems = {
    "dense": dense_search,
    "lexical": lexical_search,
}

comparison_rows = []
for system_name, search_fn in systems.items():
    rows = evaluate_retriever(BENCHMARK, search_fn, k=5)
    summary = aggregate_metrics(rows, k=5)
    comparison_rows.append(
        {
            "system": system_name,
            "recall@5": summary["recall@5"],
            "precision@5": summary["precision@5"],
            "mrr": summary["mrr"],
            "ndcg@5": summary["ndcg@5"],
        }
    )

print_rows(comparison_rows, columns=["system", "recall@5", "precision@5", "mrr", "ndcg@5"])

## Step 6 — Inspect query-level failures

Averages hide a lot. The table below shows the top-ranked documents for each query so you can see where dense retrieval helps and where it still confuses similar infrastructure themes.

In [ ]:
failure_rows = []
for item in BENCHMARK:
    dense_top = [row["doc_id"] for row in dense_search(item["query"], k=3)]
    lexical_top = [row["doc_id"] for row in lexical_search(item["query"], k=3)]
    failure_rows.append(
        {
            "qid": item["qid"],
            "query": clip(item["query"], 60),
            "relevant": ", ".join(relevant_ids(item["judgments"])),
            "dense_top3": ", ".join(dense_top),
            "lexical_top3": ", ".join(lexical_top),
        }
    )

print_rows(failure_rows, columns=["qid", "query", "relevant", "dense_top3", "lexical_top3"])

## Step 7 — Check sensitivity to k

Precision usually drops as k rises, while recall tends to rise. That tradeoff matters when you choose how many chunks to pass downstream into a generator.

In [ ]:
k_rows = []
for system_name, search_fn in systems.items():
    for k in [1, 3, 5]:
        rows = evaluate_retriever(BENCHMARK, search_fn, k=k)
        summary = aggregate_metrics(rows, k=k)
        k_rows.append(
            {
                "system": system_name,
                "k": k,
                "recall": summary["recall@{}".format(k)],
                "precision": summary["precision@{}".format(k)],
                "mrr": summary["mrr"],
                "ndcg": summary["ndcg@{}".format(k)],
            }
        )

print_rows(k_rows, columns=["system", "k", "recall", "precision", "mrr", "ndcg"])

## Step 8 — Lightweight playground

Try rewriting the query below to see how wording shifts the ranking. This is a simple way to connect retrieval metrics back to the query-transformation ideas from the earlier RAG track.

In [ ]:
custom_query = "Where is the on-route pantograph charging system mentioned?"
print("Custom query:", custom_query)
print_rows(dense_search(custom_query, k=5), columns=["doc_id", "title", "score"])
print()
print("Human expectation: D6 should rank highly because it mentions Central Station pantograph charging.")

## Recap

You now have a fully local retrieval benchmark with transparent relevance labels and from-scratch metrics. That gives you a clean foundation for the next notebooks, where we will evaluate what happens after retrieval: grounded answers, citations, and end-to-end RAG behavior.

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** design an eval dataset for a new use case. Document what changes and why.

**Exercise 2 — Build:** implement a custom metric and compare it to the one in this notebook. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** run the evaluation on a different model and analyze the differences.

## 📝 Key Takeaways

- **Learning goals** — revisit this section for reference
- **Why start here?** — revisit this section for reference
- **Step 1 — Define a small local corpus** — revisit this section for reference
- **Step 2 — Create benchmark queries and judgments** — revisit this section for reference
- **Step 3 — Run dense retrieval over the benchmark** — revisit this section for reference


## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the evals/ directory